# Mémoire — Construction d'une base ménage ECVM et comparaison économétrie / machine learning

Ce notebook prépare une base ménage à partir des modules ECVM RDC 2024, construit un indice de richesse par ACP/PCA, définit une variable binaire de pauvreté, puis compare plusieurs modèles de prédiction :

- **Logit**
- **Random Forest**
- **Gradient Boosting**
- **HistGradientBoosting**

L'organisation suit une logique de mémoire : préparation des données, construction des variables, modélisation, évaluation et interprétation.

## 1. Importation des bibliothèques

On charge ici les bibliothèques nécessaires à la préparation des données, à la visualisation et à la modélisation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report,
    roc_auc_score
)
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

import statsmodels.api as sm

## 2. Chargement des bases ECVM

Les modules utilisés dans cette analyse sont :

- `s00` : identification du ménage
- `s01` : composition du ménage
- `s02` : éducation
- `s03` : santé
- `s04a`, `s04b`, `s04c` : emploi
- `s11` : logement
- `s12` : actifs du ménage

Le chargement est effectué depuis le dossier `Base_Finale_ECVM`.

In [ ]:
folder = "Base_Finale_ECVM"

files = [
    "s00_me_rdc_2024.dta",
    "s01_me_rdc_2024.dta",
    "s02_me_rdc_2024.dta",
    "s03_me_rdc_2024.dta",
    "s04a_me_rdc_2024.dta",
    "s04b_me_rdc_2024.dta",
    "s04c_me_rdc_2024.dta",
    "s11_me_rdc_2024.dta",
    "s12_me_rdc_2024.dta"
]

data = {}
for f in files:
    data[f] = pd.read_stata(os.path.join(folder, f), convert_categoricals=False)

print("Bases chargées :")
print(list(data.keys()))

## 3. Mise en variables des modules

Cette étape rend le notebook plus lisible en affectant chaque module à une variable simple.

In [ ]:
s00 = data["s00_me_rdc_2024.dta"].copy()
s01 = data["s01_me_rdc_2024.dta"].copy()
s02 = data["s02_me_rdc_2024.dta"].copy()
s03 = data["s03_me_rdc_2024.dta"].copy()
s04a = data["s04a_me_rdc_2024.dta"].copy()
s04b = data["s04b_me_rdc_2024.dta"].copy()
s04c = data["s04c_me_rdc_2024.dta"].copy()
s11 = data["s11_me_rdc_2024.dta"].copy()
s12 = data["s12_me_rdc_2024.dta"].copy()

## 4. Construction de la base individuelle

Les modules individuels sont fusionnés sur l'identifiant :

- `grappe`
- `menage`
- `numind`

On garde uniquement les colonnes nouvelles à chaque étape afin d'éviter les conflits de noms de variables.

In [ ]:
cles_indiv = ['grappe', 'menage', 'numind']

base_indiv = s01.copy()
bases_a_merger = [s02, s03, s04a, s04b, s04c]

for i, df in enumerate(bases_a_merger, start=2):
    cols_nouvelles = [c for c in df.columns if c not in base_indiv.columns]
    df_clean = df[cles_indiv + [c for c in cols_nouvelles if c not in cles_indiv]]
    base_indiv = base_indiv.merge(df_clean, on=cles_indiv, how='left')
    print(f"Fusion terminée avec la base {i} -> dimensions : {base_indiv.shape}")

### Vérification de l'unicité de l'identifiant individuel

Une base individuelle propre doit avoir **zéro doublon** sur `grappe-menage-numind`.

In [ ]:
print("Doublons sur l'identifiant individuel :", base_indiv[cles_indiv].duplicated().sum())

## 5. Construction des premières variables ménage

À partir de la base individuelle, on agrège plusieurs caractéristiques au niveau du ménage :

- taille du ménage
- âge moyen
- part des femmes
- niveau moyen d'éducation

In [ ]:
# Reconstruction de l'âge à partir de l'année de naissance
base_indiv['age'] = 2024 - pd.to_numeric(base_indiv['s01q03aa'], errors='coerce')

# Indicateur femme
base_indiv['femme'] = (base_indiv['s01q01'] == 2).astype(int)

# Taille du ménage
taille_menage = (
    base_indiv.groupby(['grappe', 'menage'])['numind']
    .count()
    .reset_index(name='taille_menage')
)

# Âge moyen
age_moyen = (
    base_indiv.groupby(['grappe', 'menage'])['age']
    .mean()
    .reset_index(name='age_moyen')
)

# Part des femmes
part_femmes = (
    base_indiv.groupby(['grappe', 'menage'])['femme']
    .mean()
    .reset_index(name='part_femmes')
)

# Éducation moyenne
education_moyenne = (
    base_indiv.groupby(['grappe', 'menage'])['s02q29']
    .mean()
    .reset_index(name='education_moyenne')
)

base_menage = taille_menage.merge(age_moyen, on=['grappe', 'menage'], how='left')
base_menage = base_menage.merge(part_femmes, on=['grappe', 'menage'], how='left')
base_menage = base_menage.merge(education_moyenne, on=['grappe', 'menage'], how='left')

print(base_menage.shape)
base_menage.head()

## 6. Ajout des modules ménage `s00` et `s11`

Ces deux modules sont déjà au niveau ménage.  
On supprime d'abord les doublons éventuels sur l'identifiant, puis on ajoute uniquement les colonnes nouvelles.

In [ ]:
cles_menage = ['grappe', 'menage']

s00_unique = s00.drop_duplicates(subset=cles_menage)
s11_unique = s11.drop_duplicates(subset=cles_menage)

cols_nouvelles_s00 = [c for c in s00_unique.columns if c not in base_menage.columns]
s00_clean = s00_unique[cles_menage + [c for c in cols_nouvelles_s00 if c not in cles_menage]]

cols_nouvelles_s11 = [c for c in s11_unique.columns if c not in base_menage.columns]
s11_clean = s11_unique[cles_menage + [c for c in cols_nouvelles_s11 if c not in cles_menage]]

base_menage = base_menage.merge(s00_clean, on=cles_menage, how='left')
base_menage = base_menage.merge(s11_clean, on=cles_menage, how='left')

print(base_menage.shape)
print("Doublons ménage :", base_menage[cles_menage].duplicated().sum())

## 7. Transformation du module `s12` (actifs) en base ménage

Le module `s12` n'est pas directement au niveau ménage : il comporte plusieurs lignes par ménage, une ligne par article ou type de bien.

On le transforme en indicateurs agrégés :

- nombre de types d'actifs possédés
- nombre total d'actifs
- valeur totale des actifs
- âge moyen des actifs

In [ ]:
s12_small = s12[['grappe', 'menage', 's12q01', 's12q02', 's12q03', 's12q07', 's12q08']].copy()

for col in ['s12q01', 's12q02', 's12q03', 's12q07', 's12q08']:
    s12_small[col] = pd.to_numeric(s12_small[col], errors='coerce')

print("Répartition de s12q02 :")
print(s12_small['s12q02'].value_counts(dropna=False).sort_index())

# On garde uniquement les actifs possédés (1 = oui)
s12_possedes = s12_small[s12_small['s12q02'] == 1].copy()

# Préparation des variables quantitatives
s12_possedes['s12q03'] = s12_possedes['s12q03'].fillna(0)
s12_possedes['s12q08'] = s12_possedes['s12q08'].fillna(0)

nb_types_actifs = (
    s12_possedes.groupby(['grappe', 'menage'])['s12q01']
    .nunique()
    .reset_index(name='nb_types_actifs')
)

nb_total_actifs = (
    s12_possedes.groupby(['grappe', 'menage'])['s12q03']
    .sum()
    .reset_index(name='nb_total_actifs')
)

valeur_actifs = (
    s12_possedes.groupby(['grappe', 'menage'])['s12q08']
    .sum()
    .reset_index(name='valeur_totale_actifs')
)

age_moyen_actifs = (
    s12_possedes.groupby(['grappe', 'menage'])['s12q07']
    .mean()
    .reset_index(name='age_moyen_actifs')
)

s12_menage = nb_types_actifs.merge(nb_total_actifs, on=cles_menage, how='outer')
s12_menage = s12_menage.merge(valeur_actifs, on=cles_menage, how='outer')
s12_menage = s12_menage.merge(age_moyen_actifs, on=cles_menage, how='outer')

print(s12_menage.shape)
print("Doublons s12_menage :", s12_menage[cles_menage].duplicated().sum())
s12_menage.head()

### Fusion des indicateurs d'actifs avec la base ménage

In [ ]:
base_menage = base_menage.merge(s12_menage, on=cles_menage, how='left')

print(base_menage.shape)
print("Doublons base_menage :", base_menage[cles_menage].duplicated().sum())

## 8. Nettoyage des colonnes dupliquées et des variables techniques

Les fusions peuvent créer des suffixes `_x` et `_y`.  
On nettoie ces colonnes pour obtenir une base plus propre et plus lisible.

In [ ]:
cols = base_menage.columns.tolist()
cols_to_drop = []

for col in cols:
    if col.endswith('_y'):
        base_name = col[:-2]
        if base_name in cols:
            cols_to_drop.append(col)

for col in cols:
    if col.endswith('_x'):
        base_name = col[:-2]
        if base_name in cols:
            cols_to_drop.append(col)

base_menage = base_menage.drop(columns=list(set(cols_to_drop)), errors='ignore')

rename_dict = {}
for col in base_menage.columns:
    if col.endswith('_x'):
        base_name = col[:-2]
        if base_name not in base_menage.columns:
            rename_dict[col] = base_name

base_menage = base_menage.rename(columns=rename_dict)

colonnes_techniques = [
    c for c in base_menage.columns
    if c.startswith('interview__')
    or c.startswith('rejections__')
    or c.startswith('entities__')
    or c.startswith('questions__')
    or c == 'pdf'
]

base_menage = base_menage.drop(columns=colonnes_techniques, errors='ignore')

print(base_menage.shape)
print("Doublons ménage après nettoyage :", base_menage[cles_menage].duplicated().sum())

## 9. Construction des caractéristiques du chef de ménage

Le chef de ménage est identifié dans `s01` par `s01q02 == 1`.

On extrait ici :

- le sexe du chef
- l'âge du chef

In [ ]:
print("Distribution de la relation au chef :")
print(s01['s01q02'].value_counts(dropna=False).sort_index())

s01['age_calcule'] = 2024 - pd.to_numeric(s01['s01q03aa'], errors='coerce')
s01 = s01.copy()

chef = s01[s01['s01q02'] == 1].copy()
chef = chef[['grappe', 'menage', 's01q01', 'age_calcule']]

chef = chef.rename(columns={
    's01q01': 'sexe_chef',
    'age_calcule': 'age_chef'
})

print(chef.shape)
print("Doublons chef :", chef[cles_menage].duplicated().sum())
chef.head()

### Fusion des variables du chef avec la base ménage

In [ ]:
base_menage = base_menage.merge(chef, on=cles_menage, how='left')

print(base_menage.shape)
print("Doublons ménage :", base_menage[cles_menage].duplicated().sum())

## 10. Construction de l'indice de richesse par PCA

Comme l'analyse porte sur la pauvreté non monétaire, on construit un **indice de richesse** à partir :

- de variables de logement (`s11`)
- des indicateurs d'actifs (`s12`)

L'ACP/PCA est une méthode classique dans les enquêtes ménages de type DHS ou LSMS.

In [ ]:
vars_logement = [
    's11q02',   # type de logement
    's11q05',   # accès à l'eau
    's11q07',   # toilettes
    's11q08',   # électricité
    's11q14',   # matériau des murs
    's11q15',   # matériau du toit
    's11q16'    # matériau du sol
]

vars_actifs = [
    'nb_types_actifs',
    'nb_total_actifs',
    'valeur_totale_actifs',
    'age_moyen_actifs'
]

vars_pca = vars_logement + vars_actifs

X_pca = base_menage[vars_pca].copy()
X_pca = X_pca.fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_pca)

pca = PCA(n_components=1)
base_menage['indice_riche'] = pca.fit_transform(X_scaled)

base_menage['indice_riche'].describe()

### Visualisation de la distribution de l'indice de richesse

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(base_menage['indice_riche'], bins=50)
plt.title("Distribution de l'indice de richesse")
plt.xlabel("Indice de richesse")
plt.ylabel("Effectif")
plt.show()

## 11. Définition de la variable binaire de pauvreté

On retient ici une définition simple :

- `pauvre = 1` si l'indice de richesse est inférieur à la médiane
- `pauvre = 0` sinon

Cette règle permet d'obtenir une classification binaire équilibrée pour la comparaison des modèles.

In [ ]:
seuil = base_menage['indice_riche'].median()
base_menage['pauvre'] = (base_menage['indice_riche'] < seuil).astype(int)

base_menage['pauvre'].value_counts()

## 12. Sélection des variables explicatives pour la modélisation

On retient ici un ensemble restreint de variables ménage afin d'éviter un modèle trop chargé et de garder une lecture économique simple.

In [ ]:
features = [
    'taille_menage',
    'age_moyen',
    'part_femmes',
    'education_moyenne',
    'age_chef',
    'sexe_chef',
    'nb_types_actifs',
    'nb_total_actifs',
    'valeur_totale_actifs'
]

X = base_menage[features].copy()
y = base_menage['pauvre']

print(X.shape, y.shape)
X.head()

## 13. Séparation en échantillons d'entraînement et de test

La base est divisée en :

- **80 %** pour l'entraînement
- **20 %** pour l'évaluation hors échantillon

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape, X_test.shape)

## 14. Diagnostic des valeurs manquantes

Avant l'estimation, on examine le taux de valeurs manquantes.  
Les algorithmes utilisés n'acceptent pas tous les `NaN`, donc une étape d'imputation est nécessaire.

In [ ]:
print((X_train.isna().mean() * 100).sort_values(ascending=False))

## 15. Prétraitement pour la modélisation

On applique les opérations suivantes :

1. remplacement des valeurs infinies par `NaN`
2. imputation des valeurs manquantes par la médiane observée dans l'échantillon d'entraînement

La même transformation est appliquée à l'échantillon de test.

In [ ]:
X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

imputer = SimpleImputer(strategy='median')

X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test_imp = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

print(X_train_imp.isna().sum().sum(), X_test_imp.isna().sum().sum())

## 16. Estimation des modèles

On compare quatre approches :

- **Logit** : référence économétrique
- **Random Forest**
- **Gradient Boosting**
- **HistGradientBoosting**

In [ ]:
# Logit
X_train_sm = sm.add_constant(X_train_imp)
X_test_sm = sm.add_constant(X_test_imp)

logit = sm.Logit(y_train, X_train_sm).fit()
pred_logit = (logit.predict(X_test_sm) > 0.5).astype(int)
accuracy_logit = accuracy_score(y_test, pred_logit)

# Random Forest
rf = RandomForestClassifier(n_estimators=300, random_state=42)
rf.fit(X_train_imp, y_train)
pred_rf = rf.predict(X_test_imp)
accuracy_rf = accuracy_score(y_test, pred_rf)

# Gradient Boosting
gb = GradientBoostingClassifier(random_state=42)
gb.fit(X_train_imp, y_train)
pred_gb = gb.predict(X_test_imp)
accuracy_gb = accuracy_score(y_test, pred_gb)

# HistGradientBoosting
hgb = HistGradientBoostingClassifier(random_state=42)
hgb.fit(X_train_imp, y_train)
pred_hgb = hgb.predict(X_test_imp)
accuracy_hgb = accuracy_score(y_test, pred_hgb)

comparaison = pd.DataFrame({
    "Modele": ["Logit", "Random Forest", "Gradient Boosting", "HistGradientBoosting"],
    "Accuracy": [accuracy_logit, accuracy_rf, accuracy_gb, accuracy_hgb]
})

comparaison

### Résumé du modèle Logit

On affiche le résumé économétrique afin d'interpréter les signes et la significativité des coefficients.

In [ ]:
print(logit.summary())

## 17. Évaluation détaillée du meilleur modèle

Dans les résultats précédents, le meilleur modèle peut ensuite être étudié plus finement avec :

- matrice de confusion
- rapport de classification
- AUC

In [ ]:
print("Matrice de confusion (Gradient Boosting) :")
print(confusion_matrix(y_test, pred_gb))

print("\nRapport de classification (Gradient Boosting) :")
print(classification_report(y_test, pred_gb))

proba_gb = gb.predict_proba(X_test_imp)[:, 1]
auc_gb = roc_auc_score(y_test, proba_gb)
print("\nAUC Gradient Boosting :", auc_gb)

## 18. Importance des variables

L'importance des variables du modèle Gradient Boosting permet d'identifier les principaux déterminants de la pauvreté dans l'échantillon étudié.

In [ ]:
importance = pd.Series(
    gb.feature_importances_,
    index=X_train_imp.columns
).sort_values(ascending=False)

importance

In [ ]:
plt.figure(figsize=(8, 4))
importance.sort_values().plot(kind='barh')
plt.title("Importance des variables - Gradient Boosting")
plt.xlabel("Importance")
plt.ylabel("Variables")
plt.show()

## 19. Sauvegarde de la base finale

Cette cellule permet de sauvegarder la base finale utilisée dans l'analyse.

In [ ]:
base_menage.to_csv("base_menage_ecvm_propre.csv", index=False, encoding="utf-8-sig")
print("Base sauvegardée : base_menage_ecvm_propre.csv")

## 20. Conclusion du notebook

Ce notebook permet :

- de construire une base ménage propre à partir des modules ECVM
- de générer un indice de richesse par PCA
- de définir une variable binaire de pauvreté
- de comparer économétrie et machine learning
- d'identifier les principaux déterminants de la pauvreté

La suite naturelle consiste à rédiger l'interprétation économique et méthodologique des résultats dans le mémoire.